# SDTM CT Extract — Extract CDISC Test Codes from SDTM Terminology (NCI EVS)

Extract all domain-level test code / test name codelist pairs from the official
CDISC SDTM Terminology file downloaded directly from NCI EVS. Auto-discovers
all extensible codelists whose Codelist Name contains "Test Code", covering both
the `*TESTCD` convention (e.g. LBTESTCD) and the `*CD` convention used by
TA-specific and body-system codelists (e.g. CVFATSCD, MUSCTSCD).
Excludes Parameter Name-Code pairs (TSPARMCD/TSPARM) — these are study-level
metadata, not observation test codes.

## Pipeline Position

```
sdtm-test-codes/
├── notebooks/
│   ├── SDTM_CT_Extract.ipynb      ← this notebook
│   └── SDTM_CT_NCIt_Enrich.ipynb
├── downloads/
│   └── SDTM_Terminology.txt       ← cached NCI EVS source
├── interim/
│   └── SDTM_CT_Extract.csv        ← output (input for Enrich)
├── machine_actionable/
│   └── SDTM_Test_Identity.xlsx    ← produced by Enrich
└── reports/
    └── SDTM_CT_EVS_Extract_*.xlsx ← summary report
```

## Source: NCI EVS Direct Download

The NCI EVS publishes SDTM Controlled Terminology as a tab-delimited text file with an
overloaded row structure — codelist headers and code members share the same
columns. Header rows have `Codelist Code` = NaN, requiring parent-child
detection logic to extract members.

## Scope: CDISC Controlled Terminology Only

This notebook extracts **submission values** from CDISC CT codelists:
NCIt code (identity anchor), TESTCD, TEST, domain, and codelist membership.
NCIt preferred terms and UMLS enrichment are SDTM_CT_NCIt_Enrich's responsibility.

## Why All Domains?

A single NCIt concept can belong to multiple SDTM domain codelists.
Extracting all codelist pairs in one pass captures this cross-domain
membership correctly.

## Filtering Logic

The filter rule — extensible codelists whose name contains "Test Code" — comes
from a structural categorization of all 1,181 SDTM CT codelists (see
`skills/sdtm-ct-analysis/`). That analysis identified 25 semantic categories
and found that extensibility is the primary machine-actionable discriminator
separating domain-level test code codelists (56, extensible) from
instrument-specific ones (353, non-extensible). Parameter Name-Code pairs
share the same paired structure but serve a different role (study-level
metadata) and are excluded.

## Output

- `interim/SDTM_CT_Extract.csv` — full data, all domains (input for SDTM_CT_NCIt_Enrich)
- `reports/SDTM_CT_EVS_Extract_{timestamp}.xlsx` — summary report

In [ ]:
import pandas as pd
import requests
import re
from pathlib import Path
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

print('=' * 70)
print('SDTM CT EXTRACT — CDISC TEST CODES FROM NCI EVS')
print('=' * 70)
print(f"Run date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

## 1. Configuration

In [ ]:
ACTIVE_DOMAIN = 'ALL'
USE_CACHE = True

BASE_DIR = Path.cwd().parent
DOWNLOADS_DIR = BASE_DIR / 'downloads'
INTERIM_DIR = BASE_DIR / 'interim'
REPORTS_DIR = BASE_DIR / 'reports'

DOWNLOADS_DIR.mkdir(exist_ok=True)
INTERIM_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

SDTM_FILE = DOWNLOADS_DIR / 'SDTM_Terminology.txt'
SDTM_URL = 'https://evs.nci.nih.gov/ftp1/CDISC/SDTM/SDTM%20Terminology.txt'
OUTPUT_CSV = INTERIM_DIR / 'SDTM_CT_Extract.csv'

print(f'Domain filter: {ACTIVE_DOMAIN}')
print(f'Cache mode:    {"ON" if USE_CACHE else "OFF"}')
print(f'Output CSV:    {OUTPUT_CSV}')

## 2. Download / Load SDTM Terminology

In [ ]:
def fetch_file(url, local_path, description):
    if USE_CACHE and local_path.exists():
        size_kb = local_path.stat().st_size / 1024
        print(f'  Cached: {local_path.name} ({size_kb:.1f} KB)')
        return
    if not USE_CACHE and local_path.exists():
        local_path.unlink()
        print(f'  Cache OFF — re-downloading {local_path.name}')
    print(f'  Downloading {description}...', end='', flush=True)
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    local_path.write_bytes(response.content)
    print(f' {len(response.content)/1024:.1f} KB')

fetch_file(SDTM_URL, SDTM_FILE, 'SDTM Terminology from NCI EVS')

## 3. Load and Parse SDTM Terminology

In [ ]:
import os

sdtm_df = pd.read_csv(SDTM_FILE, sep='\t', dtype=str)
print(f'Loaded {len(sdtm_df):,} rows from {SDTM_FILE.name}')
print(f'Columns: {sdtm_df.columns.tolist()}')

# Source date: file modification time (versionless URL — reflects download date)
file_mtime = os.path.getmtime(SDTM_FILE)
SOURCE_DATE = datetime.fromtimestamp(file_mtime).strftime('%Y-%m-%d')
print(f'\nSource file date: {SOURCE_DATE}')

## 4. Auto-Discover Test Code / Test Name Codelist Pairs

In [ ]:
codelist_headers = sdtm_df[sdtm_df['Codelist Code'].isna()].copy()
print(f'Total codelist headers: {len(codelist_headers)}')

cl_lookup = {}
for _, row in codelist_headers.iterrows():
    short_name = str(row['CDISC Submission Value']).strip()
    cl_lookup[short_name] = {
        'ncit_code': row['Code'],
        'display_name': row['Codelist Name'],
        'extensible': str(row['Codelist Extensible (Yes/No)']).strip(),
    }

# --- Pair discovery ---
# Domain-level test code/name codelists use TWO naming conventions:
#   Pattern A: *TESTCD / *TEST   (e.g. LBTESTCD/LBTEST, VSTESTCD/VSTEST)
#   Pattern B: *CD / *           (e.g. CVFATSCD/CVFATS, MUSCTSCD/MUSCTS)
# Both share: Codelist Name contains "Test Code", extensible = Yes.
# Excluded: Parameter codelists (TSPARMCD) — study-level metadata, not tests.
#
# Discovery: Codelist Name + extensibility, pair by name substitution.

testcode_codelists = {
    sv: info for sv, info in cl_lookup.items()
    if 'Test Code' in (info['display_name'] or '')
    and info['extensible'] == 'Yes'
    and 'Parameter' not in (info['display_name'] or '')
}
print(f'Extensible Test Code codelists (excl. Parameter): {len(testcode_codelists)}')

codelist_pairs = []
orphans = []
for tc_sv in sorted(testcode_codelists):
    tc_info = testcode_codelists[tc_sv]
    tc_name = tc_info['display_name']
    tn_name_expected = tc_name.replace('Test Code', 'Test Name')

    # Find the matching Test Name codelist by display name
    tn_sv = None
    for sv, info in cl_lookup.items():
        if info['display_name'] == tn_name_expected:
            tn_sv = sv
            break

    if tn_sv is None:
        orphans.append(tc_sv)
        continue

    # Derive domain prefix from submission value
    if tc_sv.endswith('TESTCD'):
        domain = tc_sv[:-6]
    elif tc_sv.endswith('CD'):
        domain = tc_sv[:-2]
    else:
        domain = tc_sv

    domain = domain if domain else '??'
    domain_label = tc_name.replace(' Test Code', '').strip()

    codelist_pairs.append({
        'domain': domain,
        'domain_label': domain_label,
        'testcd_short': tc_sv,
        'testcd_ncit': tc_info['ncit_code'],
        'testcd_display': tc_name,
        'test_short': tn_sv,
        'test_ncit': cl_lookup[tn_sv]['ncit_code'],
        'test_display': cl_lookup[tn_sv]['display_name'],
    })

for o in orphans:
    print(f'  WARNING: {o} ({testcode_codelists[o]["display_name"]}) — no matching Test Name codelist')

print(f'\nDiscovered {len(codelist_pairs)} domain-level test code/name codelist pairs:')
print(f'{"Domain":12} {"Domain Name":50} {"TC Codelist":16} {"TN Codelist":16}')
print('-' * 96)
for p in codelist_pairs:
    print(f"{p['domain']:12} {p['domain_label']:50} {p['testcd_short']:16} {p['test_short']:16}")

## 5. Apply Domain Filter

In [ ]:
if ACTIVE_DOMAIN == 'ALL':
    active_pairs = codelist_pairs
    print(f'Extracting ALL {len(active_pairs)} codelist pairs')
else:
    active_pairs = [p for p in codelist_pairs if p['domain'] == ACTIVE_DOMAIN]
    if not active_pairs:
        available = [p['domain'] for p in codelist_pairs]
        raise ValueError(f"Domain '{ACTIVE_DOMAIN}' not found. Available: {available}")
    print(f"Extracting domain: {ACTIVE_DOMAIN} ({len(active_pairs)} pair(s))")

## 6. Extract Test Code Members

In [ ]:
test_records = []
domain_stats = []

for pair in active_pairs:
    testcd_members = sdtm_df[sdtm_df['Codelist Code'] == pair['testcd_ncit']].copy()
    test_members = sdtm_df[sdtm_df['Codelist Code'] == pair['test_ncit']].copy()

    testcd_map = testcd_members.set_index('Code')['CDISC Submission Value'].to_dict()
    test_map = test_members.set_index('Code')['CDISC Submission Value'].to_dict()

    all_codes = set(testcd_members['Code'].unique()) | set(test_members['Code'].unique())

    for ncit_code in sorted(all_codes):
        test_records.append({
            'NCIt_code': ncit_code,
            'TESTCD': testcd_map.get(ncit_code, ''),
            'TEST': test_map.get(ncit_code, ''),
            'domain': pair['domain'],
            'domain_label': pair['domain_label'],
            'codelist_testcd': pair['testcd_short'],
            'codelist_test': pair['test_short'],
            'source': 'NCI_EVS',
        })

    domain_stats.append({
        'domain': pair['domain'],
        'domain_label': pair['domain_label'],
        'testcd_codelist': pair['testcd_short'],
        'test_codelist': pair['test_short'],
        'testcd_count': len(testcd_members),
        'test_count': len(test_members),
        'ncit_codes_union': len(all_codes),
    })

df_out = pd.DataFrame(test_records)

# Validate NCIt codes (must match C + digits)
ncit_pattern = re.compile(r'^C\d+$')
bad_mask = ~df_out['NCIt_code'].apply(lambda x: bool(ncit_pattern.match(str(x))))
n_bad = bad_mask.sum()
if n_bad > 0:
    print(f'\nDATA QUALITY: {n_bad} rows with malformed NCIt codes (excluded):')
    for _, row in df_out[bad_mask].iterrows():
        msg = '  ' + str(row['NCIt_code']) + '  ' + str(row['TESTCD']).ljust(15) + str(row['TEST'])
        print(msg + '  codelist=' + str(row['codelist_testcd']))
    df_out = df_out[~bad_mask].copy()

df_out = df_out.sort_values(['domain', 'TESTCD']).reset_index(drop=True)
df_stats = pd.DataFrame(domain_stats)

print(f'Total records: {len(df_out)}')
print(f'Unique NCIt codes: {df_out["NCIt_code"].nunique()}')
print(f'\nPer-domain counts:')
print(df_stats.to_string(index=False))

## 7. Cross-Domain Membership Analysis

In [ ]:
domain_membership = df_out.groupby('NCIt_code').agg(
    domains=('domain', lambda x: sorted(x.unique())),
    n_domains=('domain', 'nunique'),
    TESTCD=('TESTCD', 'first'),
    TEST=('TEST', 'first'),
).reset_index()

multi_domain = domain_membership[domain_membership['n_domains'] > 1].copy()
multi_domain['domains_str'] = multi_domain['domains'].apply(', '.join)

print(f'NCIt codes in exactly 1 domain: {len(domain_membership) - len(multi_domain)}')
print(f'NCIt codes in multiple domains:  {len(multi_domain)}')

if len(multi_domain) > 0:
    print(f'\nCross-domain codes (first 20):')
    for _, row in multi_domain.head(20).iterrows():
        print(f"  {row['NCIt_code']:10} {row['TESTCD']:15} {row['TEST'][:35]:35} -> {row['domains_str']}")
    if len(multi_domain) > 20:
        print(f'  ... and {len(multi_domain) - 20} more')

## 7b. Codelist Subset Detection

Generic check: for each domain, is its NCIt code set a complete subset of
another domain's codelist? Detected purely from data — no hardcoded domain
rules. Adds a `Subset_Of` column to the Domain Summary report.

In [ ]:
# Build NCIt code sets per domain
domain_code_sets = df_out.groupby('domain')['NCIt_code'].apply(set).to_dict()

# For each domain, find all domains whose codelist is a strict superset
subset_of = {}
for dom, codes in domain_code_sets.items():
    supersets = []
    for other, other_codes in domain_code_sets.items():
        if other == dom:
            continue
        if codes <= other_codes and len(codes) < len(other_codes):
            supersets.append(other)
    subset_of[dom] = '; '.join(sorted(supersets)) if supersets else ''

df_stats['subset_of'] = df_stats['domain'].map(subset_of)

n_subsets = sum(1 for v in subset_of.values() if v)
print(f'Domains that are strict subsets of another: {n_subsets}')
if n_subsets > 0:
    for dom, parent in sorted(subset_of.items()):
        if parent:
            n = len(domain_code_sets[dom])
            print(f'  {dom:<10} ({n:>4} codes) subset of {parent}')

## 8. Save Interim CSV

In [ ]:
df_out.to_csv(OUTPUT_CSV, index=False)
print(f'Saved: {OUTPUT_CSV}')
print(f'  {len(df_out)} rows')
print(f'  Columns: {df_out.columns.tolist()}')
print(f'  Domains: {sorted(df_out["domain"].unique())}')

## 9. Generate Summary Report

In [ ]:
HEADER_FONT = Font(name='Arial', bold=True, size=10, color='FFFFFF')
HEADER_FILL = PatternFill('solid', fgColor='548235')
DATA_FONT = Font(name='Arial', size=10)
DATA_FILL = PatternFill('solid', fgColor='C6E0B4')
THIN_BORDER = Border(
    left=Side(style='thin', color='999999'),
    right=Side(style='thin', color='999999'),
    top=Side(style='thin', color='999999'),
    bottom=Side(style='thin', color='999999')
)
readme_title_font = Font(name='Arial', bold=True, size=14, color='548235')
readme_section_font = Font(name='Arial', bold=True, size=11, color='548235')
readme_text_font = Font(name='Arial', size=10)
readme_bold_font = Font(name='Arial', bold=True, size=10)

REPORT_TS = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_XLSX = REPORTS_DIR / f'SDTM_CT_EVS_Extract_{REPORT_TS}.xlsx'
print(f'Report: {OUTPUT_XLSX}')

In [ ]:
wb = Workbook()
ws_readme = wb.active
ws_readme.title = 'README'

n_total = len(df_out)
n_domains = df_out['domain'].nunique()
n_unique_ncit = df_out['NCIt_code'].nunique()
n_multi = len(multi_domain)
filter_label = 'ALL domains' if ACTIVE_DOMAIN == 'ALL' else f'{ACTIVE_DOMAIN} only'

readme_lines = [
    ('SDTM CONTROLLED TERMINOLOGY — TEST CODE EXTRACTION (NCI EVS)', readme_title_font),
    ('', None),
    (f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}', readme_text_font),
    (f'Source: NCI EVS SDTM Terminology (direct download)', readme_text_font),
    (f'Source date: {SOURCE_DATE} (file modification date)', readme_text_font),
    (f'Domain filter: {filter_label}', readme_text_font),
    ('', None),
    ('SCOPE', readme_section_font),
    ('', None),
    ('CDISC CT submission values only: NCIt code, TESTCD, TEST, domain, codelist.', readme_text_font),
    ('NCIt preferred terms and UMLS enrichment are added by SDTM_CT_NCIt_Enrich.', readme_text_font),
    ('', None),
    ('OVERVIEW', readme_section_font),
    ('', None),
    (f'Total records: {n_total:,} (one per NCIt_code x codelist pair)', readme_text_font),
    (f'Unique NCIt codes: {n_unique_ncit:,}', readme_text_font),
    (f'Domains: {n_domains}', readme_text_font),
    (f'Cross-domain codes (in >1 codelist): {n_multi}', readme_text_font),
    ('', None),
    ('SHEETS', readme_section_font),
    ('', None),
    ('README — this sheet', readme_text_font),
    ('Domain Summary — per-domain codelist statistics', readme_text_font),
    ('Cross-Domain Codes — NCIt codes appearing in multiple domain codelists', readme_text_font),
]

for row_idx, (text, font) in enumerate(readme_lines, start=1):
    cell = ws_readme.cell(row=row_idx, column=1, value=text)
    if font:
        cell.font = font
ws_readme.column_dimensions['A'].width = 100
ws_readme.sheet_properties.tabColor = '548235'

# --- Domain Summary ---
ws_stats = wb.create_sheet('Domain Summary')
STATS_COLS = [
    ('domain', 'Domain', 8), ('domain_label', 'Domain Name', 30),
    ('testcd_codelist', 'TESTCD Codelist', 16), ('test_codelist', 'TEST Codelist', 16),
    ('testcd_count', 'TESTCD Count', 14), ('test_count', 'TEST Count', 14),
    ('ncit_codes_union', 'NCIt Codes', 12),
    ('subset_of', 'Subset Of', 20),
]
for ci, (_, h, w) in enumerate(STATS_COLS, 1):
    c = ws_stats.cell(row=1, column=ci, value=h)
    c.font = HEADER_FONT; c.fill = HEADER_FILL
    c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    ws_stats.column_dimensions[get_column_letter(ci)].width = w
for ri, (_, row) in enumerate(df_stats.iterrows(), 2):
    for ci, (f, _, _) in enumerate(STATS_COLS, 1):
        c = ws_stats.cell(row=ri, column=ci, value=row[f])
        c.font = DATA_FONT; c.fill = DATA_FILL; c.border = THIN_BORDER
ws_stats.freeze_panes = 'A2'
ws_stats.auto_filter.ref = f'A1:{get_column_letter(len(STATS_COLS))}1'
ws_stats.sheet_properties.tabColor = '548235'

# --- Cross-Domain Codes ---
ws_xdom = wb.create_sheet('Cross-Domain Codes')
XDOM_COLS = [
    ('NCIt_code', 'NCIt C-code', 14), ('TESTCD', 'TESTCD', 14),
    ('TEST', 'TEST', 40), ('n_domains', '# Domains', 10),
    ('domains_str', 'Domains', 30),
]
for ci, (_, h, w) in enumerate(XDOM_COLS, 1):
    c = ws_xdom.cell(row=1, column=ci, value=h)
    c.font = HEADER_FONT; c.fill = HEADER_FILL
    c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    ws_xdom.column_dimensions[get_column_letter(ci)].width = w
xdom_sorted = multi_domain.sort_values('TESTCD').reset_index(drop=True)
for ri, (_, row) in enumerate(xdom_sorted.iterrows(), 2):
    for ci, (f, _, _) in enumerate(XDOM_COLS, 1):
        c = ws_xdom.cell(row=ri, column=ci, value=row[f])
        c.font = DATA_FONT; c.fill = DATA_FILL; c.border = THIN_BORDER
ws_xdom.freeze_panes = 'A2'
ws_xdom.auto_filter.ref = f'A1:{get_column_letter(len(XDOM_COLS))}1'
ws_xdom.sheet_properties.tabColor = '548235'

wb.save(OUTPUT_XLSX)
print(f'Saved: {OUTPUT_XLSX}')
print(f'  Sheets: {wb.sheetnames}')

print()
print('=' * 70)
print('COMPLETE')
print(f'  Source: {SOURCE_DATE}  Domains: {n_domains}  Records: {n_total}  Cross-domain: {n_multi}')
print(f'  CSV:  {OUTPUT_CSV}')
print(f'  XLSX: {OUTPUT_XLSX}')
print('=' * 70)